# Gold template: fact table (Snowflake)

Builds a fact table from Silver: one row per declared grain, holding only dimension keys and measures.

**Standards this notebook follows**

1. **Declared grain.** `grain` says in words what one row is; `grain_columns` are the dimension keys that make
   a row unique. The run stops if the source has two rows for one grain.
2. **No degenerate or descriptive fields.** Only dimension keys (`<NAME>_SK`) and numeric measures. Order
   numbers, statuses, names, codes and flags go in a dimension (for example `DIM_ORDER`). The rules cell
   refuses a measure that is text or is named like an identifier, and an existing table with any other column.
3. **Every key resolves.** Type 2 dimensions take the version valid on the fact's date. A miss gets the
   Unknown member, `-1`, never NULL, and the share of misses per dimension is reported.
4. **Rows that can't be placed** (a grain key that is `-1`) go to `<TARGET>_QUARANTINE`.
5. **Late-arriving dimensions.** Each run re-reads `lookback_days` of Silver; run with `full_reload` for older ones.
6. **Upsert, never overwrite.** `MERGE` on the grain keys, updating only rows whose `_ROW_HASH` changed.
7. **Referential check** after the write: every key exists in its dimension.
8. **Audit columns:** `_ROW_HASH`, `_RUN_ID`, `_LOADED_AT`. **Layers.** one database per layer: `BRONZE`, `SILVER`, `GOLD`, with keys in `KEYS`. Tables are named `<DATABASE>.<SCHEMA>.<TABLE>`.

## Who supports this notebook

This is an **enterprise template** (`gold-fact` 1.0.0, Snowflake). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-fact platform=snowflake version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "SILVER.SALES.ORDERS"
target_table = "GOLD.SALES.FACT_SALES"
grain = "One row per sales order"
grain_columns = ["order_sk"]
# One entry per dimension key in the fact:
#   fk       the column in the fact          dim      the dimension table
#   dim_sk   the dimension's key column      dim_key  the dimension column the Silver value matches
#   source   the Silver column to look up    as_of    (Type 2 only) the Silver date that picks the version
dimensions = [
    {"fk": "order_sk", "dim": "GOLD.SALES.DIM_ORDER", "dim_sk": "order_sk", "dim_key": "order_sk",
     "source": "order_sk"},
    {"fk": "customer_sk", "dim": "GOLD.SALES.DIM_CUSTOMER", "dim_sk": "customer_sk", "dim_key": "customer_dk",
     "source": "customer_sk", "as_of": "order_date"},
    {"fk": "order_date_sk", "dim": "GOLD.COMMON.DIM_DATE", "dim_sk": "date_sk", "dim_key": "date",
     "source": "order_date"},
]
measures = {"amount": "number(18,2)", "discount": "number(18,2)", "net_amount": "number(18,2)"}
watermark_column = "_loaded_at"   # Silver's load time
lookback_days = 7                 # re-read this many days of Silver so late dimension rows are picked up
max_unknown_ratio = 0.01          # warn when more than this share of rows point at an Unknown member
run_id = ""          # the task or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
import re
import uuid
from datetime import datetime, timezone

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException

session = get_active_session()
UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_$]{0,254}")
_TYPE = re.compile(r"(?i)(varchar|string|text|boolean|number\(\d{1,2}(,\s*\d{1,2})?\)|int|integer|bigint|float|double|date|timestamp_ntz|timestamp_tz)")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name, checked: parameters arrive from tasks, so nothing else can reach the SQL."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 3 or not all(_NAME.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(parts).upper()


def c(name: str) -> str:
    if not _NAME.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return name.upper()


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip().upper()


def run(sql: str) -> list:
    """Runs one statement. DML and DDL return one summary row; collect() is how Snowpark runs them."""
    return session.sql(sql).collect()


def table_exists(name: str) -> bool:
    try:
        session.table(q(name)).schema  # resolved now, so a missing table raises here
        return True
    except SnowparkSQLException:
        return False


def row_hash(columns: list, alias: str = "") -> str:
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    a = f"{alias}." if alias else ""
    parts = [f"COALESCE(TO_VARCHAR({a}{c(x)}), CHR(0))" for x in columns] or ["''"]
    return f"SHA2(ARRAY_TO_STRING(ARRAY_CONSTRUCT({', '.join(parts)}), CHR(31)), 256)"


def is_text(dtype: str) -> bool:
    return dtype.upper().split("(")[0] in ("VARCHAR", "STRING", "TEXT")


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
LOADED_AT = f"TO_TIMESTAMP_NTZ('{loaded_at}')"
print(f"Run {run_id} at {loaded_at} UTC")

## 1. Fact rules: grain, keys and measures

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
# A fact holds dimension keys and numbers you add up, at one declared grain. Nothing else.
NUMERIC = re.compile(r"(?i)(int|integer|bigint|float|double|number\(\d{1,2}(,\s*\d{1,2})?\))")
# Names that say "identifier or description": degenerate dimensions and descriptive fields.
NOT_A_MEASURE = re.compile(r"(?i).*(_id|_no|_num|_number|_code|_key|_bk|_sk|_name|_desc|_description|_status|_type|_flag|_date|_text)$")

problems = []
fks = [c(d["fk"]) for d in dimensions]
if len(set(fks)) != len(fks):
    problems.append("Each dimension key column (fk) can appear only once")
for d in dimensions:
    for field in ("fk", "dim_sk", "dim_key", "source"):
        c(d[field])
    q(d["dim"])
    if not d["fk"].lower().endswith("_sk"):
        problems.append(f"{d['fk']}: name dimension keys <name>_sk, so everyone can tell keys from measures")
grain_keys = [c(g) for g in grain_columns]
for g in grain_keys:
    if g not in fks:
        problems.append(f"Grain column {g} must be a dimension key. Put identifiers in a dimension and use its key.")
if not grain.strip() or not grain_columns:
    problems.append("Declare the grain: a sentence in grain and the keys that make a row unique in grain_columns")
for m, t in measures.items():
    c(m)
    if not NUMERIC.fullmatch(t.strip()):
        problems.append(f"{m} is {t}: a fact holds numbers you add up. Text, dates and flags belong in a dimension.")
    if NOT_A_MEASURE.fullmatch(m):
        problems.append(f"{m} looks like an identifier or a description (a degenerate field). "
                        "Move it to a dimension and keep that dimension's key here.")
if problems:
    raise ValueError("This fact breaks the fact rules:\n- " + "\n- ".join(problems))

measure_types = {c(m): checked_type(t) for m, t in measures.items()}
expected = [*fks, *measure_types, "_ROW_HASH", "_RUN_ID", "_LOADED_AT"]
if table_exists(target_table):
    extra = sorted({f.name.strip('"') for f in session.table(q(target_table)).schema.fields} - set(expected))
    if extra:
        raise ValueError(f"{target_table} has columns the fact rules don't allow: {extra}. Move them to a dimension.")
print(f"Grain: {grain} ({', '.join(grain_keys)})")

## 2. Read Silver and look up every dimension key

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
wm = c(watermark_column)
run(f"""CREATE TABLE IF NOT EXISTS {q(target_table)} (
    {", ".join([*[f"{f} NUMBER(10,0) NOT NULL" for f in fks], *[f"{m} {t}" for m, t in measure_types.items()],
                "_ROW_HASH VARCHAR(64)", "_RUN_ID VARCHAR", "_LOADED_AT TIMESTAMP_NTZ"])})""")
since = "" if full_reload else (
    f"WHERE s.{wm} >= (SELECT DATEADD(day, -{int(lookback_days)}, MAX(_LOADED_AT)) FROM {q(target_table)}) "
    f"OR NOT EXISTS (SELECT 1 FROM {q(target_table)})")

joins, keys = [], []
for i, d in enumerate(dimensions):
    a = f"d{i}"
    on = f"{a}.{c(d['dim_key'])} = s.{c(d['source'])} AND {a}.{c(d['dim_sk'])} <> {UNKNOWN_SK}"
    if d.get("as_of"):
        day = f"TO_DATE(s.{c(d['as_of'])})"
        on += f" AND {day} >= {a}.VALID_FROM AND {day} < {a}.VALID_TO"
    joins.append(f"LEFT JOIN {q(d['dim'])} {a} ON {on}")
    keys.append(f"COALESCE({a}.{c(d['dim_sk'])}, {UNKNOWN_SK}) AS {c(d['fk'])}")

run(f"""CREATE OR REPLACE TEMPORARY TABLE FACT_STAGED AS
SELECT x.*, {row_hash([*fks, *measure_types], 'x')} AS _ROW_HASH, '{run_id}' AS _RUN_ID, {LOADED_AT} AS _LOADED_AT
FROM (
    SELECT {", ".join(keys)}, {", ".join(f"CAST(s.{m} AS {t}) AS {m}" for m, t in measure_types.items())}
    FROM {q(source_table)} s
    {chr(10).join(joins)}
    {since}
) x""")

## 3. Quarantine rows that can't be placed, check the grain, report Unknowns

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
unplaced = " OR ".join(f"{g} = {UNKNOWN_SK}" for g in grain_keys)
cols = ", ".join(expected)
run(f"""CREATE TABLE IF NOT EXISTS {q(target_table)}_QUARANTINE AS
SELECT {cols}, CAST(NULL AS VARCHAR) AS _REASON FROM FACT_STAGED WHERE 1 = 0""")
rejected = run(f"""INSERT INTO {q(target_table)}_QUARANTINE ({cols}, _REASON)
SELECT {cols}, 'a grain dimension row is missing' FROM FACT_STAGED WHERE {unplaced}""")
if rejected and rejected[0][0]:
    print(f"{rejected[0][0]} rows couldn't be placed at the grain; they are in {target_table}_QUARANTINE")
run(f"DELETE FROM FACT_STAGED WHERE {unplaced}")

grain_list = ", ".join(grain_keys)
duplicates = run(f"SELECT {grain_list} FROM FACT_STAGED GROUP BY {grain_list} HAVING COUNT(*) > 1 LIMIT 5")
if duplicates:
    raise RuntimeError(f"Two or more source rows share one grain ({grain_list}): {duplicates}. "
                       "The grain is wrong, or Silver has duplicates.")

stats = run(f"SELECT COUNT(*) AS ROW_COUNT, {', '.join(f'COUNT_IF({f} = {UNKNOWN_SK}) AS {f}' for f in fks)} FROM FACT_STAGED")[0]
for f in fks:
    ratio = stats[f] / stats["ROW_COUNT"] if stats["ROW_COUNT"] else 0
    if ratio > max_unknown_ratio:
        print(f"Warning: {ratio:.1%} of rows have {f} = {UNKNOWN_SK} (Unknown). Load that dimension first, "
              "or check the business keys.")

## 4. Upsert into the fact

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=step-4 | Enterprise-managed: change the Parameters cell, not this one.
on = " AND ".join(f"t.{g} = s.{g}" for g in grain_keys)
result = run(f"""MERGE INTO {q(target_table)} AS t
USING FACT_STAGED AS s
ON {on}
WHEN MATCHED AND t._ROW_HASH <> s._ROW_HASH THEN
    UPDATE SET {", ".join(f"t.{n} = s.{n}" for n in expected if n not in grain_keys)}
WHEN NOT MATCHED THEN
    INSERT ({cols}) VALUES ({", ".join(f"s.{n}" for n in expected)})""")
print(f"Fact MERGE: {result[0].as_dict() if result else 'done'}")

## 5. Check the grain and every key

In [ ]:
# kratosforge:standard template=gold-fact platform=snowflake version=1.0.0 cell=step-5 | Enterprise-managed: change the Parameters cell, not this one.
dupes = run(f"SELECT {grain_list} FROM {q(target_table)} GROUP BY {grain_list} HAVING COUNT(*) > 1 LIMIT 5")
orphans = {}
for d in dimensions:
    missing = run(f"""SELECT COUNT(*) FROM {q(target_table)} f
    WHERE NOT EXISTS (SELECT 1 FROM {q(d['dim'])} x WHERE x.{c(d['dim_sk'])} = f.{c(d['fk'])})""")[0][0]
    if missing:
        orphans[d["fk"]] = missing
if dupes or orphans:
    raise RuntimeError(f"{target_table} breaks a fact rule. Duplicate grain: {dupes}. Keys missing from their dimension: {orphans}")
print(f"{target_table}: grain holds and every key is in its dimension")